In [5]:
import stim
import re
import os
from pyzx import *
import matplotlib.pyplot as plt
import networkx as nx
from pyvis.network import Network

def stim_qasm_comply(qasm: str) -> str:
    q = qasm
    q = re.sub(r'def\s+rx\(qubit q0\)\s*\{[^}]*\}\n+', '', q)
    q = re.sub(r'rx\s*\(\s*q\[(\d+)\]\s*\)\s*;', r'h q[\1];', q)
    q = re.sub(r'reset\s+q\[(\d+)\];', '', q)
    return q



In [6]:
# s = f"test_{i}"
# print(f"Testing universal representation for {s}")
n = 5
k = 1
# tableau = stim.Tableau.random(n)
bug_code = [stim.PauliString("-XYZZ_"), stim.PauliString("+ZYX_Z"), stim.PauliString("+ZZYY_"), stim.PauliString("-Y_X_Y"), stim.PauliString("-XZYZY")]


tableau = stim.Tableau.from_stabilizers(bug_code)
stabilizers = []

for i in range (n - k):
    stabilizers.append(stim.PauliString(f"Z{i}") * stim.PauliString(n+k))

for i in range(n-k, n): 
    stabilizers.append(stim.PauliString(f"Z{i}*Z{i+k}") * stim.PauliString(n+k)) 
    stabilizers.append(stim.PauliString(f"X{i}*X{i+k}") * stim.PauliString(n+k)) 


state = stim.TableauSimulator()
state.set_state_from_stabilizers(stabilizers)
state.do_tableau(tableau, list(range(n)))
t = state.current_inverse_tableau().inverse()
qasm_random = t.to_circuit(method="graph_state").to_qasm(open_qasm_version=3)       
pyzx_circ = Circuit.from_qasm(stim_qasm_comply(qasm_random))
g = pyzx_circ.to_graph()
input_state = "0"*(n+k)
g.apply_state(input_state)
g.set_inputs(g.outputs()[n:n+k])
# inputs = list(g.inputs())
# g = GraphState(g)
ugr1 = graph_to_ZXCF(g).graph
# draw(g.get_graph())
draw(ugr1)
t1 = tensorfy(g)  

qasm_random2 = tableau.to_circuit(method = "elimination").to_qasm(open_qasm_version=3)
pyzx_circ2 = Circuit.from_qasm(qasm_random2)
input_state2 = "0"*(n - k) + "/"*k
g2 = pyzx_circ2.to_graph()
g2.apply_state(input_state2)
# inputs = list(g2.inputs())
# g2 = GraphState(g2)
ugr2 = graph_to_ZXCF(g2).graph
# draw(g2.get_graph())
t2 = tensorfy(g2)
draw(ugr2)

print(compare_tensors(t1,t2))

if not compare_tensors(t1,t2):    
    print(tableau.to_stabilizers())



CJI


CJI


True


In [8]:
file_path = f"./test_graphs/test_9.qasm"
n = 5
k = 1
with open(file_path, "r") as f:
    qasm_random = f.read()
pyzx_circ = Circuit.from_qasm(qasm_random)
g = pyzx_circ.to_graph()
input_state = "0"*(n-k) + "/"*k
g.apply_state(input_state)
draw(g)
g = GraphState(g)
draw(g.get_graph())
t1 = tensorfy(g.get_graph())
g.to_canonical_form()
t2 = tensorfy(g.get_graph())
draw(g.get_graph())
compare_tensors(t1, t2)

CJI


True